# 4.86: Dominant eigenvalue

$det ( A - \lambda I ) = 0$

$\left| \begin{bmatrix}
1& 2& 3& 4\\
5& 6& 7& 8\\
9& 0& 1& 2\\
3& 4& 5& 6
\end{bmatrix} - \begin{bmatrix}
\lambda& 0&0& 0\\
0& \lambda& 0& 0\\
0& 0& \lambda& 0\\
0& 0& 0& \lambda
\end{bmatrix}\right| = 0 $

In [1]:
import numpy as np


def power_method(A, tol=1e-10, max_iter=10_000, seed=2024):
    """Dominant eigenpair and residual of a square matrix."""
    A = np.asarray(A, dtype=float)
    rng = np.random.default_rng(seed)
    x = rng.standard_normal(A.shape[0])
    x /= np.linalg.norm(x)
    eigenvalue = 0.0
    for iteration in range(1, max_iter + 1):
        Ax = A @ x
        eigenvalue = x @ Ax
        residual = np.linalg.norm(Ax - eigenvalue * x)
        if residual < tol:
            break
        x = Ax / np.linalg.norm(Ax)
    return eigenvalue, x, residual, iteration


A = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 0, 1, 2],
              [3, 4, 5, 6]], dtype=float)
eigenvalue, eigenvector, residual, iterations = power_method(A)
print(f"dominant eigenvalue: {eigenvalue:.10f}")
print(f"eigenvector: {eigenvector}")
print(f"iterations: {iterations}, residual: {residual:.2e}")
print(f"numpy reference: {np.linalg.eigvals(A).real.max():.10f}")


dominant eigenvalue: 15.8167749046
eigenvector: [0.29492336 0.75736962 0.25016262 0.52614649]
iterations: 16, residual: 1.72e-11
numpy reference: 15.8167749045


# 4.90: Deflation

In [2]:
def deflation(A, tol=1e-10, seed=2024):
    """Wielandt deflation: remove the dominant eigenpair and iterate again."""
    A = np.asarray(A, dtype=float)
    lambda1, v1, _, _ = power_method(A, tol=tol, seed=seed)

    # Left eigenvector for the same eigenvalue; normalize so w1^T v1 = 1.
    _, w1, _, _ = power_method(A.T, tol=tol, seed=seed + 1)
    w1 = w1 / (w1 @ v1)

    deflated = A - lambda1 * np.outer(v1, w1)
    lambda2, v2, residual, _ = power_method(deflated, tol=tol, seed=seed + 2)
    return lambda1, v1, lambda2, v2, residual


lambda1, v1, lambda2, v2, residual = deflation(A)
print(f"largest eigenvalue: {lambda1:.10f}")
print(f"next dominant eigenvalue: {lambda2:.10f}")
print(f"residual of the second pair: {residual:.2e}")
print(f"numpy eigenvalues: {np.linalg.eigvals(A)}")


largest eigenvalue: 15.8167749046
next dominant eigenvalue: -2.3539489506
residual of the second pair: 7.43e-11
numpy eigenvalues: [ 1.58167749e+01+0.j -2.35394895e+00+0.j  1.10386854e-15+0.j
  5.37174046e-01+0.j]


# 4.91: Probability vectors

## a) 
Given a transition matrix $A$, the state vector at day $k$, denoted as $s_k$.
Then, the difference equation $s_{k+1} = A s_k$.

We have the initial state vector $s_0$, and the state vector after one day $s_1$ is $s_1 = A s_0$

Then, applying the transition matrix $A$ again to $s_1$ to get the state vector after two days $s_2$ is 
$$ s_2 = A s_1 = A (A s_0) = A^2 s_0 $$

Then, if we continue this process iteratively for $n$ days,
$$ s_n = A(A s_{n-1}) = A(A (A s_{n-2})) = \cdots = A (A^{n-1} s_0) = A^n s_0 $$

Thus, $ s_n = A^n s_0$

In [3]:
A = np.array([
    [0.9, 0.3, 0, 0, 0],
    [0.05, 0.7, 0.4, 0, 0],
    [0.025, 0, 0.6, 0.6, 0],
    [0.015, 0, 0, 0.4, 0.8],
    [0.01, 0, 0, 0, 0.2]])
s_0 = np.array([[1],[0],[0],[0],[0]])
n = 5
s_n = np.linalg.matrix_power(A, n) @ s_0
print(f"s{n} = \n{s_n}")

s5 = 
[[0.7016085 ]
 [0.17282925]
 [0.08443313]
 [0.03187808]
 [0.00925105]]


## b) 

In [4]:
A = np.array([
    [0.9, 0.3, 0, 0, 0],
    [0.05, 0.7, 0.4, 0, 0],
    [0.025, 0, 0.6, 0.6, 0],
    [0.015, 0, 0, 0.4, 0.8],
    [0.01, 0, 0, 0, 0.2]])

_, steady_state, residual, iterations = power_method(A)
steady_state = steady_state / steady_state.sum()   # normalize to a probability vector
print(f"steady-state probabilities: {steady_state}")
print(f"residual: {residual:.2e}, iterations: {iterations}")
print(f"column sums of A: {A.sum(axis=0)}")


steady-state probabilities: [0.66115702 0.22038567 0.08264463 0.02754821 0.00826446]
residual: 8.85e-11, iterations: 59
column sums of A: [1. 1. 1. 1. 1.]


## c) 

## d)
The normalization involves dividing each element of the steady-state vector by the sum of all its elements to ensure that the total probability is 1. Each value in this output vector represents the long-term probability in a corresponding flood stage.

66% that the river will be at normal flow, which average daily flow is below 90,000 $ft^3/sec$ (Stage 0).<br>
22% that the river will be at action level. Average daily flow is between 90,000 cfs and 124,000 cfs (Stage 1).<br>
8% that the river will experience minor flooding (Stage 2). Average daily flow is between 124,000 cfs and 146,000 cfs.<br>
3% of moderate flood (Stage 3). Average daily flow is between 146,000 cfs and 170,000 cfs.<br>
0.8% of extreme flood (Stage 4). Average daily flow is above 170,000 cfs.

# SVD Question

## a)
In the full SVD, $\Sigma$ has the same dimensions as the original matrix A. That means, if $A$ is $m\times n$, where $m\neq n$ then $\Sigma$ is also $m \times n$.

The dimensions of $U$ and $V$ in the full SVD also needs to satisfy $A = U\Sigma V^T$, which remains the dimensions of $m\times n$.

Therefore, $U$ must be $m\times m$ matrix, and $V$ must be $n\times n$ matrix, and both of them are orthogonal matrices.

## b)
Consider an $m\times n$ matrix A.

We can try to reduce the matrices to a smaller set by focusing on the non-zero singular values.

Since $U$ is an $m\times m$ matrix where $m > n$. Thus we can reduce $U$ to an $m\times n$ matrix. Then, $\Sigma$ can be reduced to an $n\times n$ diagonal matrix only contains the non-zero singular values instead of the $m\times n$ form. Similar idea can be applied to $V^T$, but as $V^T$ is an $n\times n$ orthogonal matrix already, it can remain the same through the reduction process.

Using reduced matrices to reconstruct A will give $U_{m\times n} \Sigma_{n\times n} V^T_{n\times n}$, which also return an $m\times n$ reconstructed matrix A.

## c)

In [5]:
def mySVD(A):
    """SVD from the eigendecomposition of A^T A; returns U, Sigma, V."""
    A = np.asarray(A, dtype=float)
    m, n = A.shape
    eigenvalues, eigenvectors = np.linalg.eigh(A.T @ A)
    order = np.argsort(eigenvalues)[::-1]
    V = eigenvectors[:, order]
    sigma = np.sqrt(np.clip(eigenvalues[order], 0, None))
    tol = np.sqrt(np.finfo(float).eps) * sigma[0]
    rank = int(np.sum(sigma > tol))

    if rank:
        U, _ = np.linalg.qr(A @ V[:, :rank], mode="complete")
        for i in range(rank):
            if U[:, i] @ (A @ V[:, i]) < 0:
                U[:, i] *= -1
    else:
        U = np.eye(m)

    Sigma = np.zeros((m, n))
    Sigma[:rank, :rank] = np.diag(sigma[:rank])
    return U, Sigma, V


A_svd = np.array([[3.0, -2.0],
                  [1.0, -4.0],
                  [2.0, 1.0]])
U, Sigma, V = mySVD(A_svd)
print(f"reconstruction error: {np.linalg.norm(A_svd - U @ Sigma @ V.T):.2e}")
print(f"singular values: {np.diag(Sigma)}")
print(f"numpy: {np.linalg.svd(A_svd, compute_uv=False)}")


reconstruction error: 7.02e-16
singular values: [5.12173063 2.96105984]
numpy: [5.12173063 2.96105984]
